In [ ]:
from pathlib import Path
import re

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from sklearn.neighbors import NearestNeighbors


In [ ]:
root_dir = Path('/Volumes/B/Projects/2025Project/SpatialAnalysis/Project_ICC')
base_dir = root_dir / 'Data/ST/Samples'
patient_info_path = root_dir / 'Table/Patient_Info.csv'
anno_path = root_dir / 'Data/ST/inferCNV/inferCNV_result/subcluster_anno.tsv'
smoothed_anno_path = root_dir / 'Data/ST/inferCNV/inferCNV_result/anno_CNV_subcluster_smoothed.csv'
hex_fig_dir = root_dir / 'Figure/ST/InferCNV/Spatial_Hex'
cell_fig_dir = root_dir / 'Figure/ST/InferCNV/Spatial_CNV_DataIntegration'

spatial_key = 'spatial_um'
clone_col = 'Clone'
plot_col = 'Clone_plot'
epithelial_only = True
show_plots = True
spot_size = 10
figsize = (6, 6)


In [ ]:
def parse_sampleid(value):
    match = re.fullmatch(r'Pt(\d+)-(\d+)([a-zA-Z])?', str(value).strip())
    if match is None:
        return (10**9, 10**9, 10**9)
    patient, section, suffix = match.groups()
    suffix_order = 0 if suffix is None else ord(suffix.lower()) - ord('a') + 1
    return (int(patient), int(section), suffix_order)

patient_info = pd.read_csv(patient_info_path, dtype=str)
sample_info = patient_info.loc[patient_info['Type'].eq('Tumor')].dropna(
    subset=['SampleName', 'SampleID']
).copy()
sample_info['sort_key'] = sample_info['SampleID'].map(parse_sampleid)
sample_info = sample_info.sort_values(['sort_key', 'SampleID', 'SampleName'])
samples = sample_info['SampleName'].drop_duplicates().tolist()
name2id = sample_info.set_index('SampleName')['SampleID'].to_dict()

# To process a subset, replace the sample list here.
# samples = ['Sample01']
print(f'Found {len(samples)} tumor samples.')


In [ ]:
CLONE_COLORS = {'A': '#f3b0b3', 'B': '#5e8fac', 'C': '#f5d05a', 'D': '#f08b1a', 'E': '#238444', 'F': '#82539e', 'G': '#ac7447', 'H': '#802268FF' 'unassigned': '#f0f0f0',}


In [ ]:
def smooth_cnv_knn_balanced(adata, label_col: str='CNV_subcluster', out_col: str='CNV_subcluster_smooth', basis: str='spatial_um', ignore_labels=('conflict', 'NA', 'nan', 'None', '', 'unassigned'), k_local: int=10, frac_local: float=0.6, min_valid_local: int=5, max_same_local: int=3, k_broad: int=80, min_valid_broad: int=20, protect_same_broad_count: int=6, protect_same_broad_frac: float=0.08, broad_protect_mode: str='and', global_protect_frac: float | None=0.15, global_protect_min_count: int | None=50, rare_label_min_count: int | None=10, rare_k: int=15, rare_min_valid: int=1, rare_maj_frac: float=0.5, rare_exclude_self_label: bool=True, force_rare_label: bool=True, max_dist_local: float | None=None, max_dist_broad: float | None=None, max_dist_rare: float | None=None, n_iters: int=1, k_search: int | None=None, group_col: str | None=None, changed_col: str | None='CNV_smooth_changed', reason_col: str | None='CNV_smooth_reason'):
 
    if basis not in adata.obsm:
        raise KeyError(f"adata.obsm['{basis}'] does not exist")
    if label_col not in adata.obs:
        raise KeyError(f"adata.obs['{label_col}'] does not exist")
    XY_all = np.asarray(adata.obsm[basis], dtype=float)
    if XY_all.ndim != 2 or XY_all.shape[1] < 2:
        raise ValueError(f"adata.obsm['{basis}'] must have shape (n, 2) or (n, >=2); received: {XY_all.shape}")
    XY_all = XY_all[:, :2]
    raw = adata.obs[label_col].astype('object')
    labels0 = raw.astype(str).to_numpy()
    labels0[raw.isna().to_numpy()] = '<NA>'
    ignore = set(map(str, ignore_labels)) | {'<NA>'}

    def is_valid_label(x):
        return str(x) not in ignore
    labels_out = labels0.copy()
    n_all = XY_all.shape[0]
    changed_all = np.zeros(n_all, dtype=bool)
    reason_all = np.array(['unchanged'] * n_all, dtype=object)
    if group_col is not None:
        if group_col not in adata.obs:
            raise KeyError(f"adata.obs['{group_col}'] does not exist")
        groups = []
        for _, idx in adata.obs.groupby(group_col, observed=False).indices.items():
            groups.append(np.asarray(idx, dtype=int))
    else:
        groups = [np.arange(n_all)]

    def broad_protected(broad_labs, self_lab):
        same_broad_count = int(np.sum(broad_labs == self_lab))
        same_broad_frac = same_broad_count / len(broad_labs)
        count_hit = same_broad_count >= protect_same_broad_count
        frac_hit = same_broad_frac >= protect_same_broad_frac
        if broad_protect_mode == 'or':
            return count_hit or frac_hit
        elif broad_protect_mode == 'and':
            return count_hit and frac_hit
        else:
            raise ValueError("broad_protect_mode must be 'and' or 'or'")
    for gidx in groups:
        if len(gidx) <= 1:
            continue
        XY = XY_all[gidx]
        labels = labels_out[gidx].copy()
        n = len(gidx)
        changed = np.zeros(n, dtype=bool)
        reason = np.array(['unchanged'] * n, dtype=object)
        max_k_need = max(k_local, k_broad, rare_k)
        if k_search is None:
            k_search_use = min(n, max(max_k_need * 5, 200))
        else:
            k_search_use = min(n, int(k_search))
        nn = NearestNeighbors(n_neighbors=min(k_search_use + 1, n), metric='euclidean')
        nn.fit(XY)
        neigh_dist, neigh_idx = nn.kneighbors(XY, return_distance=True)

        def get_valid_neighbors(i, k_need, max_dist=None, exclude_label: str | None=None):
            valid = []
            for j, d in zip(neigh_idx[i], neigh_dist[i]):
                if j == i:
                    continue
                if max_dist is not None and d > max_dist:
                    continue
                lj = labels[j]
                if not is_valid_label(lj):
                    continue
                if exclude_label is not None and str(lj) == str(exclude_label):
                    continue
                valid.append(j)
                if len(valid) >= k_need:
                    break
            return valid
        for _ in range(n_iters):
            new_labels = labels.copy()
            valid_labels_for_count = [x for x in labels if is_valid_label(x)]
            group_label_counts = pd.Series(valid_labels_for_count).value_counts()
            n_valid_group = int(group_label_counts.sum())
            label_frac_map = {}
            if n_valid_group > 0:
                label_frac_map = (group_label_counts / n_valid_group).to_dict()
            for i in range(n):
                self_lab = labels[i]
                if not is_valid_label(self_lab):
                    continue
                self_lab_str = str(self_lab)
                self_label_count = int(group_label_counts.get(self_lab_str, 0))
                if force_rare_label and rare_label_min_count is not None and (self_label_count < rare_label_min_count):
                    exclude_label = self_lab_str if rare_exclude_self_label else None
                    rare_idx = get_valid_neighbors(i, k_need=rare_k, max_dist=max_dist_rare, exclude_label=exclude_label)
                    if len(rare_idx) >= rare_min_valid:
                        rare_labs = labels[rare_idx]
                        vc_rare = pd.Series(rare_labs).value_counts()
                        rare_maj_lab = vc_rare.index[0]
                        new_labels[i] = rare_maj_lab
                        changed[i] = True
                        reason[i] = 'force_rare_label'
                        continue
                    else:
                        reason[i] = 'force_rare_no_target'
                        continue
                if not force_rare_label and rare_label_min_count is not None and (self_label_count < rare_label_min_count):
                    exclude_label = self_lab_str if rare_exclude_self_label else None
                    rare_idx = get_valid_neighbors(i, k_need=rare_k, max_dist=max_dist_rare, exclude_label=exclude_label)
                    if len(rare_idx) >= rare_min_valid:
                        rare_labs = labels[rare_idx]
                        vc_rare = pd.Series(rare_labs).value_counts()
                        rare_maj_lab = vc_rare.index[0]
                        rare_maj_frac_obs = vc_rare.iloc[0] / len(rare_labs)
                        if rare_maj_frac_obs >= rare_maj_frac:
                            new_labels[i] = rare_maj_lab
                            changed[i] = True
                            reason[i] = 'rare_label'
                            continue
                self_global_frac = float(label_frac_map.get(self_lab_str, 0.0))
                protect_by_frac = global_protect_frac is not None and self_global_frac >= global_protect_frac
                protect_by_count = global_protect_min_count is not None and self_label_count > global_protect_min_count
                if protect_by_frac or protect_by_count:
                    reason[i] = 'global_protected'
                    continue
                local_idx = get_valid_neighbors(i, k_need=k_local, max_dist=max_dist_local, exclude_label=None)
                if len(local_idx) < min_valid_local:
                    continue
                local_labs = labels[local_idx]
                vc_local = pd.Series(local_labs).value_counts()
                maj_lab = vc_local.index[0]
                maj_frac = vc_local.iloc[0] / len(local_labs)
                same_local_count = int(np.sum(local_labs == self_lab))
                if maj_lab == self_lab:
                    continue
                if maj_frac < frac_local:
                    continue
                if same_local_count > max_same_local:
                    continue
                broad_idx = get_valid_neighbors(i, k_need=k_broad, max_dist=max_dist_broad, exclude_label=None)
                if len(broad_idx) >= min_valid_broad:
                    broad_labs = labels[broad_idx]
                    if broad_protected(broad_labs, self_lab):
                        continue
                new_labels[i] = maj_lab
                changed[i] = True
                reason[i] = 'local_isolated'
            labels = new_labels
        labels_out[gidx] = labels
        changed_all[gidx] = changed
        reason_all[gidx] = reason
    adata.obs[out_col] = pd.Categorical(labels_out)
    if changed_col is not None:
        adata.obs[changed_col] = changed_all
    if reason_col is not None:
        adata.obs[reason_col] = pd.Categorical(reason_all)
    return adata


In [ ]:
anno = pd.read_csv(anno_path, sep='\t', dtype=str).set_index('Hex_ID')
hex_fig_dir.mkdir(parents=True, exist_ok=True)
all_updates = []

for sid in samples:
    show_name = name2id.get(sid, sid)
    sample_dir = base_dir / sid
    h5ad_file = sample_dir / 'hexagon/to_inferCNV_36um.h5ad'
    if not h5ad_file.exists():
        print(f'Skipping missing input: {h5ad_file}')
        continue

    print(f'Annotating and smoothing {sid}')
    adata = sc.read_h5ad(h5ad_file)
    adata.obs['CNV_subcluster'] = anno.reindex(adata.obs_names)['CNV_subcluster']
    adata.obs['CNV_subcluster'] = adata.obs['CNV_subcluster'].fillna('unassigned')
    adata = smooth_cnv_knn_balanced(
        adata,
        label_col='CNV_subcluster',
        out_col='CNV_subcluster_smooth',
        basis='spatial_um',
        ignore_labels=('unassigned',),
        rare_label_min_count=10,
        rare_k=10,
        rare_min_valid=1,
        rare_exclude_self_label=True,
        force_rare_label=True,
        global_protect_frac=0.15,
        global_protect_min_count=50,
        k_local=15,
        min_valid_local=3,
        frac_local=0.5,
        max_same_local=5,
        k_broad=100,
        min_valid_broad=10,
        protect_same_broad_count=10,
        protect_same_broad_frac=0.05,
        broad_protect_mode='and',
        n_iters=1,
    )
    update = adata.obs[['CNV_subcluster', 'CNV_subcluster_smooth']].copy()
    update.index.name = 'Hex_ID'
    all_updates.append(update)
    adata.write_h5ad(sample_dir / 'hexagon/add_InferCNV_adata_36um.h5ad')

    fig, axes = plt.subplots(1, 2, figsize=(11, 5), constrained_layout=True)
    try:
        fig.suptitle(show_name, fontsize=14, y=1.05)
        for ax, label_col in zip(axes, ['CNV_subcluster', 'CNV_subcluster_smooth']):
            sc.pl.spatial(
                adata, color=label_col, basis=spatial_key,
                spot_size=50, show=False, palette=CLONE_COLORS, ax=ax,
            )
            ax.set_title('')
            ax.set_aspect('equal')
            ax.axis('off')
        fig.savefig(hex_fig_dir / f'{show_name}_infercnv_subclusters_mapping.pdf', bbox_inches='tight')
        fig.savefig(hex_fig_dir / f'{show_name}_infercnv_subclusters_mapping.png', dpi=300, bbox_inches='tight')
        if show_plots:
            plt.show()
    finally:
        plt.close(fig)
    del adata

if all_updates:
    updates_df = pd.concat(all_updates, axis=0)
    anno_smooth = anno.copy()
    anno_smooth['CNV_subcluster_smooth'] = updates_df['CNV_subcluster_smooth']
    smoothed_anno_path.parent.mkdir(parents=True, exist_ok=True)
    anno_smooth.to_csv(smoothed_anno_path, index=True)
    print(f'Saved {smoothed_anno_path}')
else:
    print('No bin-level inputs were processed; the annotation CSV was not written.')


Part 2 Plot existing cell-level clone labels 

In [ ]:
def load_plot_data(h5ad_path):
    source = sc.read_h5ad(h5ad_path, backed='r')
    try:
        required = [clone_col] + (['CellType'] if epithelial_only else [])
        obs = source.obs[required].copy()
        xy = np.asarray(source.obsm[spatial_key], dtype=float).copy()
    finally:
        source.file.close()

    labels = obs[clone_col].astype('string').str.strip()
    invalid = labels.isna() | labels.str.lower().isin(
        ['', 'nan', 'none', 'na', '<na>', 'conflict', 'unassigned']
    )
    labels = labels.mask(invalid, 'unassigned')
    if epithelial_only:
        is_epithelial = (
            obs['CellType'].astype('string').str.strip().eq('Epithelial').fillna(False)
        )
        labels = labels.where(is_epithelial, 'unassigned')

    unknown = set(labels) - set(CLONE_COLORS)
    if unknown:
        raise ValueError(f'Add colors for these clone labels: {sorted(unknown)}')
    categories = [label for label in CLONE_COLORS if label in set(labels)]
    obs[plot_col] = pd.Categorical(labels, categories=categories)
    plot_data = ad.AnnData(obs=obs)
    plot_data.obsm[spatial_key] = xy[:, :2]
    return plot_data


def plot_one_sample(sid, png_dir):
    input_path = base_dir / sid / 'integrated/cell_adata.h5ad'
    plot_data = load_plot_data(input_path)
    fig, ax = plt.subplots(1, 1, figsize=figsize, constrained_layout=True)
    try:
        sc.pl.spatial(
            plot_data,
            color=plot_col,
            palette=CLONE_COLORS,
            basis=spatial_key,
            spot_size=spot_size,
            show=False,
            legend_loc=None,
            ax=ax,
            title='',
        )
        ax.set_aspect('equal')
        ax.axis('off')
        fig.savefig(png_dir / f'{sid}.png', dpi=300, bbox_inches='tight')
        if show_plots:
            plt.show()
    finally:
        plt.close(fig)


In [ ]:
cell_fig_dir.mkdir(parents=True, exist_ok=True)
for sid in samples:
    print(f'Plotting stored cell-level clone labels for {sid}')
    plot_one_sample(sid, cell_fig_dir)
